# Pemrosesan Teks dan Klasifikasi Multikelas

Catatan ini menguraikan integrasi antara teknik ekstraksi fitur dari data teks (*Natural Language Processing*) dengan arsitektur algoritma klasifikasi multikelas (kasus di mana target prediksi memiliki lebih dari dua kategori).

Fokus utama pada modul ini adalah mengubah teks mentah menjadi matriks numerik menggunakan pembobotan *Term Frequency-Inverse Document Frequency* (TF-IDF), melatih model yang optimal untuk ruang fitur berdimensi tinggi seperti *Multinomial Naive Bayes* dan *Logistic Regression*, serta mengevaluasi performa model menggunakan metrik spesifik multikelas.

#### **Tujuan Pembelajaran**
* Mampu merepresentasikan data teks bebas menjadi vektor numerik menggunakan `TfidfVectorizer`.
* Memahami konsep matematis pembobotan TF-IDF untuk menonjolkan kata kunci yang representatif.
* Menerapkan algoritma *Multinomial Naive Bayes* yang sangat efisien untuk klasifikasi teks multikelas.
* Mengevaluasi hasil prediksi multikelas menggunakan *Confusion Matrix* serta metrik *Macro Average* dan *Weighted Average*.
* Mengintegrasikan proses transformasi teks dan pemodelan ke dalam arsitektur `Pipeline`.

In [38]:
# Memuat pustaka komputasi numerik dan struktur data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Memuat pustaka ekstraksi fitur teks
from sklearn.feature_extraction.text import TfidfVectorizer

# Memuat modul pemodelan dan prapemrosesan
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression

# Memuat metrik evaluasi klasifikasi
from sklearn.metrics import classification_report, confusion_matrix

# Konfigurasi parameter visualisasi
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

print("Pustaka untuk Pemrosesan Teks dan Klasifikasi Multikelas berhasil dimuat.")

Pustaka untuk Pemrosesan Teks dan Klasifikasi Multikelas berhasil dimuat.


## 1. Definisi Dataset Teks Multikelas

Sebagai demonstrasi, kita akan menyimulasikan sebuah korpus data teks pendek (*toy dataset*) yang terdiri dari tiga kategori kelas (Multikelas): **Teknologi**, **Olahraga**, dan **Kesehatan**. Dalam praktiknya, teks ini dapat berupa ulasan produk, kategori berita, atau tiket keluhan pelanggan.

In [39]:
# Definisi Korpus Teks (Fitur X) dan Label Kategori (Target y)
dokumen_teks = [
    "spesifikasi laptop baru ini sangat cepat dengan ram besar",
    "perangkat keras komputer ini mudah panas saat digunakan",
    "smartphone terbaru memiliki kamera dengan resolusi tinggi",
    "pertandingan sepak bola semalam berakhir seri tanpa gol",
    "pemain basket itu memenangkan medali emas di turnamen",
    "tim nasional gagal lolos ke babak final kejuaraan dunia",
    "dokter menyarankan pasien untuk istirahat dan minum vitamin",
    "virus baru ini membutuhkan penanganan medis segera di rumah sakit",
    "pasokan obat obatan di apotek mulai menipis akibat pandemi"
]

label_kelas = [
    "Teknologi", "Teknologi", "Teknologi",
    "Olahraga", "Olahraga", "Olahraga",
    "Kesehatan", "Kesehatan", "Kesehatan"
]

# Konversi ke dalam Pandas DataFrame
df_teks = pd.DataFrame({"Teks": dokumen_teks, "Kategori": label_kelas})

# Pemisahan himpunan latih dan uji
X_train_txt, X_test_txt, y_train_txt, y_test_txt = train_test_split(
    df_teks["Teks"], df_teks["Kategori"], test_size=0.33, random_state=42, stratify=df_teks["Kategori"]
)

display(df_teks)

,Teks,Kategori
0,spesifikasi laptop baru ini sangat cepat denga...,Teknologi
1,perangkat keras komputer ini mudah panas saat ...,Teknologi
2,smartphone terbaru memiliki kamera dengan reso...,Teknologi
3,pertandingan sepak bola semalam berakhir seri ...,Olahraga
4,pemain basket itu memenangkan medali emas di t...,Olahraga
5,tim nasional gagal lolos ke babak final kejuar...,Olahraga
6,dokter menyarankan pasien untuk istirahat dan ...,Kesehatan
7,virus baru ini membutuhkan penanganan medis se...,Kesehatan
8,pasokan obat obatan di apotek mulai menipis ak...,Kesehatan


## 2. Ekstraksi Fitur Teks menggunakan TF-IDF

Algoritma *machine learning* beroperasi menggunakan matriks numerik, sehingga teks harus diubah menjadi angka (Vektorisasi). Daripada sekadar menghitung jumlah frekuensi kemunculan kata (*Bag-of-Words*), kita menggunakan **TF-IDF**.

Rumusan matematis TF-IDF memberikan penalti pada kata yang terlalu sering muncul di semua dokumen (seperti konjungsi) dan memberikan bobot tinggi pada kata spesifik yang mendefinisikan sebuah dokumen:

$$W(t, d) = TF(t, d) \times \log\left(\frac{N}{DF(t)}\right)$$

*(Di mana $TF$ adalah frekuensi term dalam dokumen, $N$ adalah total dokumen, dan $DF$ adalah frekuensi dokumen yang mengandung term tersebut).*

In [40]:
# Inisialisasi TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer()

# Melatih dan mentransformasi data latih
X_train_tfidf = tfidf_vectorizer.fit_transform(X_train_txt)

print(f"Total Kosakata (Vocabulary) yang dipelajari: {len(tfidf_vectorizer.vocabulary_)} kata")
print(f"Dimensi Matriks Sparse Latih: {X_train_tfidf.shape}")

# Menampilkan 10 kata pertama dalam kosakata
print("Contoh Fitur Kata (Tokens):", tfidf_vectorizer.get_feature_names_out()[:10])

Total Kosakata (Vocabulary) yang dipelajari: 48 kata
Dimensi Matriks Sparse Latih: (6, 48)
Contoh Fitur Kata (Tokens): ['babak' 'baru' 'basket' 'besar' 'cepat' 'dan' 'dengan' 'di' 'digunakan'
 'dokter']


## 3. Pemodelan Multikelas terintegrasi Pipeline

Untuk klasifikasi teks, **Multinomial Naive Bayes** (*MultinomialNB*) adalah algoritma fundamental yang sangat efisien. Model ini dirancang khusus untuk data yang direpresentasikan sebagai frekuensi atau *count* (termasuk pembobotan fraksional seperti TF-IDF), serta kebal terhadap fenomena *Curse of Dimensionality* yang sering terjadi pada data teks.

Kita akan membungkus tahapan TF-IDF dan algoritma Naive Bayes ke dalam sebuah **Pipeline** untuk mencegah *data leakage* dan menyederhanakan kode.

In [41]:
# Mengonstruksi Arsitektur Pipeline
pipe_teks = Pipeline([
    ("vektorisasi_tfidf", TfidfVectorizer()),
    ("klasifikasi_nb", MultinomialNB())
])

# Melatih arsitektur pipeline pada data teks mentah
pipe_teks.fit(X_train_txt, y_train_txt)

# Melakukan inferensi pada data uji
prediksi_teks = pipe_teks.predict(X_test_txt)

print("=== Pengujian Prediksi Multikelas ===")
for teks, aktual, prediksi in zip(X_test_txt, y_test_txt, prediksi_teks):
    print(f"Teks Aktual : '{teks}'")
    print(f"Kelas Asli  : {aktual} | Prediksi Model : {prediksi}\n")

=== Pengujian Prediksi Multikelas ===
Teks Aktual : 'pertandingan sepak bola semalam berakhir seri tanpa gol'
Kelas Asli  : Olahraga | Prediksi Model : Kesehatan

Teks Aktual : 'pasokan obat obatan di apotek mulai menipis akibat pandemi'
Kelas Asli  : Kesehatan | Prediksi Model : Olahraga

Teks Aktual : 'smartphone terbaru memiliki kamera dengan resolusi tinggi'
Kelas Asli  : Teknologi | Prediksi Model : Teknologi



## 4. Evaluasi Performa Multikelas

Dalam skenario multikelas, evaluasi menjadi lebih kompleks dibandingkan sekadar menghitung akurasi absolut. Kita wajib meninjau metrik berikut dari `classification_report`:
* **Macro Average:** Menghitung rata-rata metrik (seperti F1-Score) untuk setiap kelas secara independen, lalu merata-ratakannya. Sangat berguna untuk memastikan kelas minoritas tidak diabaikan oleh model.
* **Weighted Average:** Menghitung rata-rata dengan memberikan bobot berdasarkan proporsi sampel aktual dari masing-masing kelas.

In [42]:
# Mencetak Laporan Klasifikasi Komprehensif
print("=== Laporan Klasifikasi Multikelas ===")
print(classification_report(y_test_txt, prediksi_teks))

# Membangun Matriks Konfusi (Confusion Matrix)
label_unik = pipe_teks.classes_
matriks_konfusi = confusion_matrix(y_test_txt, prediksi_teks, labels=label_unik)

df_matriks = pd.DataFrame(
    matriks_konfusi,
    index=[f"Aktual {label}" for label in label_unik],
    columns=[f"Prediksi {label}" for label in label_unik]
)

print("=== Matriks Konfusi ===")
display(df_matriks)

=== Laporan Klasifikasi Multikelas ===
              precision    recall  f1-score   support

   Kesehatan       0.00      0.00      0.00         1
    Olahraga       0.00      0.00      0.00         1
   Teknologi       1.00      1.00      1.00         1

    accuracy                           0.33         3
   macro avg       0.33      0.33      0.33         3
weighted avg       0.33      0.33      0.33         3

=== Matriks Konfusi ===


,Prediksi Kesehatan,Prediksi Olahraga,Prediksi Teknologi
Aktual Kesehatan,0,1,0
Aktual Olahraga,1,0,0
Aktual Teknologi,0,0,1


## Kesimpulan

Berdasarkan implementasi pemrosesan teks multikelas di atas, dapat ditarik beberapa simpulan teknis:
1. **Representasi TF-IDF** terbukti krusial untuk menyeimbangkan dominasi kata umum, memungkinkan model mengidentifikasi terminologi esensial yang membedakan satu kategori dengan kategori lainnya.
2. Penggunaan **Pipeline** menyederhanakan kode dari himpunan raw data langsung ke prediksi akhir, yang mana secara mutlak mencegah vektorisator mempelajari probabilitas frekuensi dokumen (*Document Frequency*) dari data uji.
3. Algoritma komputasi probabilitas seperti **Multinomial Naive Bayes** maupun model linier (**Logistic Regression**) merupakan standar industri (*baseline*) yang sangat kuat untuk menangani masalah kategorisasi teks multikelas sebelum mengadopsi model *Deep Learning* yang memakan komputasi tinggi.